# Batched Graph Classification

*Hard · from [mlcode](https://mlcode.tech/problems/torch-graph-batching)*

Classify whole graphs. To train fast, put many small graphs into one batch,
as graph libraries do.

Each graph is a pair `(x, edge_index)`, with node features `x` of shape
`(n_i, f)` and edges of shape `(2, e_i)`. Row 0 of `edge_index` holds the
source nodes and row 1 the target nodes. The task is to tell **trees** from
graphs that **contain a cycle**. All node features are `1`, so the model
must use the structure.

Write three functions:

1. `collate(graphs)` joins a list of graphs into one big graph with no
   edges between the parts. Concatenate the features. Add an offset to each
   graph's node ids, equal to the number of nodes before it. Also return a
   `batch` vector that gives the graph index of each node.
2. `sum_aggregate(x, edge_index)` sends each source node's features to its
   target and adds them up: `out[i] = sum of x[j] over edges j -> i`. Use
   `index_add`. Do not build an adjacency matrix.
3. `train_graph_classifier(graphs, labels)` trains a GNN and returns a
   function that takes a list of graphs and returns their labels.

### Function signature

```python
def collate(graphs):                    # -> (x, edge_index, batch)
def sum_aggregate(x, edge_index):       # -> (n, f)
def train_graph_classifier(graphs, labels):   # -> predict(graphs)
```

### Example

**Input**

```
g0 = (torch.ones(2, 1), torch.tensor([[0, 1], [1, 0]]))
g1 = (torch.ones(3, 1), torch.tensor([[0, 2], [2, 0]]))
collate([g0, g1])
```

**Output**

```
x          = ones of shape (5, 1)
edge_index = tensor([[0, 1, 2, 4],
                     [1, 0, 4, 2]])
batch      = tensor([0, 0, 1, 1, 1])
```

**Reasoning**

Graph 0 has 2 nodes, so graph 1's node ids move up by 2. Its edge `0 -> 2`
becomes `2 -> 4`. The first two nodes belong to graph 0 and the next three
to graph 1.

### Constraints and notes

- `collate` returns float features, an int64 `edge_index` of shape
  `(2, E)` and an int64 `batch` of shape `(N,)`.
- `sum_aggregate` gives zeros for a node with no incoming edges. If an edge
  is listed twice, it counts twice. Gradients must flow to `x`.
- `labels` is an int64 tensor of `0` (tree) and `1` (has a cycle). The
  returned function must give int64 labels of shape `(G,)`.
- Use **sum** pooling over the nodes of each graph, with `index_add` and the
  `batch` vector. Mean pooling loses the number of nodes.
- On held-out graphs, at least 90% of the predictions must be correct. A
  correct model gives the same predictions when the nodes of a graph are
  numbered in a different order.

---

Write your solution in the cell below, then run the grading cell at the bottom. Your work is recorded against the same account as the site, so a pass here shows there.

Keep the solution in the function you were given. The grading cell saves your definitions and the lines around them back to the site, not the whole session, so anything you type to poke about is left behind rather than ending up in your saved answer.

In [ ]:
#@title Connect this notebook to mlcode { display-mode: "form" }
#@markdown Paste the code from the problem page, then run this cell.
CODE = ""  #@param {type:"string"}

SITE = "https://mlcode.tech"
SLUG = "torch-graph-batching"

import json, urllib.request, urllib.error

def _mlc_get(path):
    with urllib.request.urlopen(SITE + path, timeout=30) as r:
        return r.read()

# The harness is fetched rather than pasted in, so the cases run here exactly
# as they would on the site.
for _mlc_n in ("harness.py", "notebook.py"):
    with open("/content/" + _mlc_n, "wb") as _mlc_fh:
        _mlc_fh.write(_mlc_get("/" + _mlc_n))

try:
    _mlc_meta = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))
    print("connected:", _mlc_meta["title"])
    print("torch:", __import__("torch").__version__)
except urllib.error.HTTPError as e:
    print("could not connect:", json.loads(e.read()).get("error", e.reason))
    print("Get a fresh code from the problem page; they last three hours.")


In [ ]:
import torch
from torch import nn


def collate(graphs):
    """Join a list of (x, edge_index) graphs into one disconnected graph.

    returns: (x of shape (N, f), edge_index of shape (2, E), batch of shape (N,))
    """
    pass


def sum_aggregate(x, edge_index):
    """out[i] = sum of x[j] over the edges j -> i, with index_add.

    x:          float tensor of shape (n, f)
    edge_index: int64 tensor of shape (2, E), row 0 sources, row 1 targets
    returns: float tensor of shape (n, f)
    """
    pass


def train_graph_classifier(graphs, labels):
    """Train a GNN to tell trees (0) from graphs with a cycle (1).

    graphs: list of (x, edge_index)
    labels: int64 tensor of shape (G,)
    returns: a function from a list of graphs to int64 labels of shape (G,)
    """
    pass

In [ ]:
#@title Grade this notebook { display-mode: "form" }
#@markdown Runs the hidden tests and records the attempt on the site.
import json, sys, time, urllib.request, urllib.error

sys.path.insert(0, "/content")
import harness

# The tests are fetched now rather than stored in the notebook, so a reader
# cannot read them before attempting the problem.
_mlc_tests = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))["tests"]

# `In` is the execution history, not the current cells: a wrong first attempt,
# a cell that errored and three re-runs are all still in it. Saving that would
# put something on the site that does not even replay. Rebuild the source from
# what is actually defined now instead, and keep the history only as a fallback
# for a solution written at module level.
def _mlc_collect():
    import ast, inspect, types

    # This notebook's own form cells are not part of anyone's answer.
    _cells = [c for c in In[1:] if not c.lstrip().startswith("#@title")]

    _imports, _seen = [], set()
    _consts = dict()
    for _cell in _cells:
        for _line in _cell.split("\n"):
            _t = _line.strip()
            if (_t.startswith("import ") or _t.startswith("from ")) and _t not in _seen:
                _seen.add(_t)
                _imports.append(_t)
        try:
            _tree = ast.parse(_cell)
        except SyntaxError:
            continue
        # Module level constants the functions below may close over. Last
        # assignment wins, because an edited cell is run again.
        for _node in _tree.body:
            if isinstance(_node, ast.Assign) and len(_node.targets) == 1:
                _target = _node.targets[0]
                if isinstance(_target, ast.Name):
                    _text = ast.get_source_segment(_cell, _node)
                    if _text and len(_text) <= 300:
                        # A plain literal can go above the definitions, where a
                        # default argument can still see it. Anything else may
                        # call one of them, so it has to come after.
                        try:
                            ast.literal_eval(_node.value)
                            _lit = True
                        except Exception:
                            _lit = False
                        _consts[_target.id] = (_text, _lit)

    _defs, _defined = [], set()
    for _name, _obj in list(globals().items()):
        if _name.startswith("_mlc_") or _name in ("In", "Out"):
            continue
        if isinstance(_obj, (types.FunctionType, type)) and getattr(_obj, "__module__", "") == "__main__":
            try:
                _lead = inspect.getcomments(_obj) or ""
                _defs.append(_lead + inspect.getsource(_obj).rstrip())
                _defined.add(_name)
            except (OSError, TypeError):
                pass

    # Nothing defined means the solution is not shaped like a function, so fall
    # back to what was typed rather than returning an empty file.
    if not _defs:
        return "\n\n".join(_cells)

    _early, _late = [], []
    for _name, _pair in _consts.items():
        if _name in _defined or _name.startswith("_mlc_") or _name in ("CODE", "SITE", "SLUG"):
            continue
        # What is kept is the line of source, not the value it produced, so how
        # large the value is does not matter. The length bound above is what
        # keeps a pasted-in data literal out.
        (_early if _pair[1] else _late).append(_pair[0])

    _blocks = ["\n".join(_imports), "\n".join(_early),
               "\n\n\n".join(_defs), "\n".join(_late)]
    return "\n\n\n".join(b for b in _blocks if b.strip()) + "\n"

_mlc_source = _mlc_collect()

_mlc_started = time.time()
_mlc_results = harness.run_tests(_mlc_tests, dict(globals()), 60, _mlc_source, "submit")
_mlc_elapsed = int((time.time() - _mlc_started) * 1000)

_mlc_passed = sum(1 for r in _mlc_results if r["passed"])
for _mlc_r in _mlc_results:
    print(("  pass  " if _mlc_r["passed"] else "  FAIL  ") + _mlc_r["name"])
    if not _mlc_r["passed"] and _mlc_r["message"]:
        print("        " + _mlc_r["message"].replace("\n", "\n        "))
print(f"\n{_mlc_passed}/{len(_mlc_results)} passed")

_mlc_body = json.dumps({
    "code": CODE, "slug": SLUG, "passedCount": _mlc_passed,
    "totalCount": len(_mlc_results), "runtimeMs": _mlc_elapsed, "source": _mlc_source,
}).encode()
_mlc_req = urllib.request.Request(SITE + "/api/colab", data=_mlc_body,
                              headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(_mlc_req, timeout=30) as r:
        print("recorded on mlcode:", "solved" if json.load(r)["passed"] else "attempt saved")
except urllib.error.HTTPError as e:
    print("could not record:", json.loads(e.read()).get("error", e.reason))
